# GraphRAG Optimizer Loop for GNN-based Bottleneck Prediction

This notebook implements a **GraphRAG optimization loop** that:
1. Uses a trained HeteroGAT to predict slack and step time
2. Converts graphs to structured prompts for an LLM
3. Asks the LLM for bottleneck suggestions
4. Applies what-if modifications to the graph
5. Re-runs the GNN to evaluate improvements
6. Computes metrics (ΔT improvement, success rate, Recall@1)

In [1]:
!pip install -q torch-geometric llama-index llama-index-llms-cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, w

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


In [2]:
import re
import os
import json
import pickle
import copy
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv, Linear, global_mean_pool
from typing import Dict, List, Tuple, Any, Optional

# LlamaIndex + Cohere
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import CompletionResponse

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

import warnings
warnings.filterwarnings('ignore')

Using device: cuda


## **Cohere Model Setup**
`command-a-03-2025`

In [4]:
from google.colab import userdata
import cohere
COHERE_API_KEY = userdata.get("cohereAPI")
llm = Cohere(
    model="command-a-03-2025",
    api_key=COHERE_API_KEY,
    temperature=0.3
)

## **HeteroGAT**

In [5]:
import torch
import numpy as np

try:
    from numpy.core.multiarray import scalar as numpy_scalar
except ImportError:
    from numpy._core.multiarray import scalar as numpy_scalar

torch.serialization.add_safe_globals([numpy_scalar])

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from typing import Tuple

class HeteroGATLayer(nn.Module):
    """Single HeteroGAT layer with optional directed message passing."""

    def __init__(self, in_dim, out_dim, heads, directed=False, dropout=0.1):
        super().__init__()
        self.directed = directed
        assert out_dim % heads == 0, f'out_dim ({out_dim}) must be divisible by heads ({heads})'
        head_dim = out_dim // heads
        self.gat_fwd = GATConv(in_dim, head_dim, heads=heads,
                                dropout=dropout, add_self_loops=False)
        if not directed:
            self.gat_rev = GATConv(in_dim, head_dim, heads=heads,
                                    dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x, edge_index):
        """
        x          : node features (N, in_dim)
        edge_index : (2, E) — directed edges src→dst (ancestor→descendant)
        """
        out_fwd = self.gat_fwd(x, edge_index)

        if self.directed:
            out = out_fwd
        else:
            rev_edge_index = edge_index.flip(0)
            out_rev = self.gat_rev(x, rev_edge_index)
            out = out_fwd + out_rev

        return self.norm(F.elu(out))


class HeteroGAT(nn.Module):
    """
    Multi-layer HeteroGAT with multi-task output.

    directed=False → standard bidirectional (associational baseline)
    directed=True  → topologically-ordered directed MP (causal inductive bias)
    """

    def __init__(self, in_dim, hidden_dim, num_layers, heads,
                 directed=False, dropout=0.2):
        super().__init__()
        self.directed = directed
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim

        # Input projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        # GAT layers
        self.layers = nn.ModuleList([
            HeteroGATLayer(hidden_dim, hidden_dim, heads, directed, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

        # Head 1: per-node slack prediction
        self.slack_head = nn.Sequential(
        nn.Linear(hidden_dim, hidden_dim // 2),
        nn.ELU(),
        nn.Linear(hidden_dim // 2, 1)
        )

        # Head 2: graph-level step time prediction
        self.step_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ELU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index, batch=None):
        """
        Returns:
            slack_pred : (N,) per-node slack
            step_pred  : (B,) per-graph step time
        """
        h = F.elu(self.input_proj(x))
        h = self.dropout(h)

        for layer in self.layers:
            h = layer(h, edge_index)
            h = self.dropout(h)

        slack_pred = self.slack_head(h).squeeze(-1)   # (N,)

        # Graph readout for step time
        if batch is not None and batch.numel() > 0:
            h_graph = global_mean_pool(h, batch)       # (B, hidden)
        else:
            h_graph = h.mean(dim=0, keepdim=True)      # (1, hidden)
        step_pred = self.step_head(h_graph).squeeze(-1)  # (B,)

        return slack_pred, step_pred


def load_model(checkpoint_path: str, device: torch.device) -> Tuple[nn.Module, dict]:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    config = checkpoint['config']
    print(f"Checkpoint config keys: {config}")

    model = HeteroGAT(
        in_dim=config['in_dim'],
        hidden_dim=config['hidden_dim'],
        num_layers=config.get('num_layers_bidir', 3),  # fixed key
        heads=config.get('heads', 4),
        directed=False,
        dropout=config.get('dropout', 0.2)
    )
    model.load_state_dict(checkpoint['bidir_state'])
    model.to(device)
    model.eval()
    return model, config

In [12]:
import gdown
# Load the actual model
# MODEL_PATH = "https://drive.google.com/file/d/17bcGF86lMxr4udCh_ytHiur7vYNq2JVO/view?usp=drive_link"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Download the model from Google Drive
MODEL_ID = "17bcGF86lMxr4udCh_ytHiur7vYNq2JVO"
MODEL_PATH = "heterogat_v2.pt"

gdown.download(f'https://drive.google.com/uc?id={MODEL_ID}', MODEL_PATH, quiet=False)

model, model_config = load_model(MODEL_PATH, device)
print(f"Model loaded: {model_config}")
print(f"Directed mode: {model.directed} (should be False)")

Downloading...
From: https://drive.google.com/uc?id=17bcGF86lMxr4udCh_ytHiur7vYNq2JVO
To: /content/heterogat_v2.pt
100%|██████████| 260k/260k [00:00<00:00, 3.59MB/s]


Checkpoint config keys: {'in_dim': 7, 'hidden_dim': 64, 'num_layers_bidir': 3, 'num_layers_dir': 5, 'heads': 4, 'dropout': 0.2, 'lambda_rank': 0.056}
Model loaded: {'in_dim': 7, 'hidden_dim': 64, 'num_layers_bidir': 3, 'num_layers_dir': 5, 'heads': 4, 'dropout': 0.2, 'lambda_rank': 0.056}
Directed mode: False (should be False)


**Inferencing the Model**

In [13]:
# Update inference function to call forward correctly
def run_inference(graph):
    """Run inference on a single HeteroData graph."""
    graph = graph.to(device)
    x = graph['op'].x
    edge_index = graph['op', 'depends_on', 'op'].edge_index
    # No batch argument -> single graph treated inside forward
    with torch.no_grad():
        slack_pred, step_pred = model(x, edge_index, batch=None)
    return slack_pred.cpu().numpy(), step_pred.item()

In [14]:
def hetero_to_nx(graph: HeteroData) -> nx.DiGraph:
    """Convert HeteroData to NetworkX DiGraph. Node IDs: 'op_i'."""
    G = nx.DiGraph()
    num_nodes = graph['op'].x.size(0)
    for i in range(num_nodes):
        duration = graph['op'].x[i, 0].item()
        G.add_node(f'op_{i}', duration=duration, features=graph['op'].x[i].cpu().numpy())
    edge_index = graph['op', 'depends_on', 'op'].edge_index.cpu().numpy()
    for src, dst in zip(edge_index[0], edge_index[1]):
        G.add_edge(f'op_{src}', f'op_{dst}')
    return G

def compute_cpm_slack(G: nx.DiGraph) -> Dict[str, float]:
    """Classic critical path method slack computation."""
    topo_order = list(nx.topological_sort(G))
    est = {node: 0.0 for node in G.nodes}
    for node in topo_order:
        for pred in G.predecessors(node):
            est[node] = max(est[node], est[pred] + G.nodes[pred]['duration'])
    lst = {node: float('inf') for node in G.nodes}
    sink_nodes = [n for n in G.nodes if G.out_degree(n) == 0]
    project_duration = max(est[n] + G.nodes[n]['duration'] for n in sink_nodes)
    for node in sink_nodes:
        lst[node] = project_duration - G.nodes[node]['duration']
    for node in reversed(topo_order):
        for succ in G.successors(node):
            lst[node] = min(lst[node], lst[succ] - G.nodes[node]['duration'])
    slack = {node: lst[node] - est[node] for node in G.nodes}
    return slack

**Graph to Prompt**

In [15]:
def graph_to_prompt(graph: HeteroData, slack_pred: np.ndarray) -> str:
    """Build a structured prompt that includes node slacks and dependencies."""
    G_nx = hetero_to_nx(graph)
    node_lines = []
    for i, node in enumerate(G_nx.nodes()):
        slack = slack_pred[i]
        node_lines.append(f"- {node}: predicted slack = {slack:.4f} (lower slack = more critical)")
    nodes_str = "\n".join(node_lines)

    edge_lines = [f"{src} -> {dst}" for src, dst in G_nx.edges()]
    edges_str = "\n".join(edge_lines)

    prompt = f"""You are an expert in hardware accelerator design and graph optimization.
The following directed acyclic graph (DAG) represents operations in a computation pipeline.
Each node has a predicted slack value (lower is more critical). Slack indicates how much delay can be added without affecting the overall step time.

Nodes and their predicted slack (lower = more critical):
{nodes_str}

Dependencies (edges):
{edges_str}

Based on the predicted slacks, identify the **top 3 bottlenecks** (most critical nodes) and provide **actionable optimizations** to reduce the overall step time.
Optimizations can include: reducing execution time of a node, parallelizing independent nodes, altering dependencies, etc.

Return your answer as a JSON object with exactly the following format:
{{
  "top_bottlenecks": ["node_id1", "node_id2", "node_id3"],
  "suggestions": [
    "suggestion for node_id1",
    "suggestion for node_id2",
    "suggestion for node_id3"
  ]
}}
Do not include any additional text outside the JSON.
"""
    return prompt

**LLM Integration using LlamaIndex + Cohere**

In [16]:
cohere_client = cohere.ClientV2(api_key=COHERE_API_KEY)

def query_llm(prompt: str, temperature: float = 0.3) -> Dict[str, Any]:
    """Send prompt to Cohere's native chat API with JSON mode enforced."""
    # 1. Call the V2 chat endpoint
    response = cohere_client.chat(
        model="command-a-03-2025",               # Your selected model
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        response_format={"type": "json_object"} # Guarantees a valid JSON output
    )

    # 2. Extract the content. With JSON mode enabled, no parsing is needed.
    content = response.message.content[0].text

    # 3. Parse the guaranteed JSON string
    return json.loads(content)

**What-If-Analysis**

In [17]:
## 7. What-If Engine
def apply_what_if(graph: HeteroData, suggestion: str) -> HeteroData:
    new_graph = copy.deepcopy(graph)
    sugg_lower = suggestion.lower()

    # Pattern 1: "reduce ... node op_X ... by Y%"
    reduce_match = re.search(r'reduce.*node\s+(\S+).*?by\s+(\d+)%', sugg_lower)
    if reduce_match:
        node_id = reduce_match.group(1)
        percent = float(reduce_match.group(2)) / 100.0
        if node_id.startswith('op_'):
            idx = int(node_id.split('_')[1])
            new_graph['op'].x[idx, 0] *= (1 - percent)
            return new_graph

    # Pattern 2: "remove dependency between op_X and op_Y"
    dep_match = re.search(r'remove dependency between (\S+)\s+and\s+(\S+)', sugg_lower)
    if dep_match:
        src, dst = dep_match.group(1), dep_match.group(2)
        if src.startswith('op_') and dst.startswith('op_'):
            src_idx = int(src.split('_')[1])
            dst_idx = int(dst.split('_')[1])
            ei = new_graph['op', 'depends_on', 'op'].edge_index
            keep = ~((ei[0] == src_idx) & (ei[1] == dst_idx))
            new_graph['op', 'depends_on', 'op'].edge_index = ei[:, keep]
            return new_graph

    # Fallback: if we can't parse, reduce the duration of the first node mentioned in suggestion
    # (just to have some effect)
    any_node = re.search(r'op_\d+', suggestion)
    if any_node:
        node_id = any_node.group()
        idx = int(node_id.split('_')[1])
        # Reduce by a small default percentage
        new_graph['op'].x[idx, 0] *= 0.95
        return new_graph

    return new_graph

def evaluate_suggestion(original_graph: HeteroData, suggestion: str, original_step: float) -> Tuple[float, HeteroData]:
    mod_graph = apply_what_if(original_graph, suggestion)
    _, new_step = run_inference(mod_graph)
    delta_percent = (original_step - new_step) / original_step * 100  # positive = improvement
    return delta_percent, mod_graph


def recall_at_1(pred_slack: np.ndarray, gt_slack_dict: Dict[str, float]) -> bool:
    pred_min_idx = np.argmin(pred_slack)
    gt_min_node = min(gt_slack_dict, key=gt_slack_dict.get)
    gt_min_idx = int(gt_min_node.split('_')[1])
    return pred_min_idx == gt_min_idx

def compute_metrics(graph: HeteroData, pred_slack: np.ndarray, suggestions: List[str], original_step: float) -> Dict:
    nx_graph = hetero_to_nx(graph)
    gt_slack = compute_cpm_slack(nx_graph)
    recall = recall_at_1(pred_slack, gt_slack)

    improvements = []
    for sugg in suggestions[:3]:
        delta, _ = evaluate_suggestion(graph, sugg, original_step)
        improvements.append(delta)
    success = sum(1 for d in improvements if d > 0) >= 2
    avg_improvement = np.mean(improvements) if improvements else 0.0

    return {
        "recall_at_1": recall,
        "improvements": improvements,
        "success": success,
        "avg_improvement_pct": avg_improvement
    }

In [19]:
# ## 9. Batch Processing
import gdown

DATASET_ID = "1ZMnj0_rmXvLSFrNqYfMSPmxMEojhc_MH"
DATASET_PATH = "graphs_v2.pkl"
gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}', DATASET_PATH, quiet=False)

with open(DATASET_PATH, 'rb') as f:
    all_graphs = pickle.load(f)
print(f"Loaded {len(all_graphs)} graphs")

NUM_GRAPHS = min(20, len(all_graphs))
graphs_subset = all_graphs[:NUM_GRAPHS]

results = []
for idx, graph in enumerate(graphs_subset):
    print(f"Processing graph {idx+1}/{NUM_GRAPHS}...")
    slack_pred, step_pred = run_inference(graph)
    prompt = graph_to_prompt(graph, slack_pred)
    try:
        llm_out = query_llm(prompt, temperature=0.3)
        suggestions = llm_out.get("suggestions", [])
        bottlenecks = llm_out.get("top_bottlenecks", [])
        if idx == 0:  # debug first graph
            print(f"Sample suggestion: {suggestions[0] if suggestions else 'None'}")
    except Exception as e:
        print(f"LLM query failed: {e}")
        suggestions = []
        bottlenecks = []
    metrics = compute_metrics(graph, slack_pred, suggestions, step_pred)
    results.append({
        "graph_id": idx,
        "predicted_bottleneck": f"op_{np.argmin(slack_pred)}",
        "llm_bottlenecks": bottlenecks,
        "step_original": step_pred,
        "suggestions": suggestions,
        "improvements": metrics["improvements"],
        "avg_improvement_pct": metrics["avg_improvement_pct"],
        "success": metrics["success"],
        "recall_at_1": metrics["recall_at_1"]
    })
df_results = pd.DataFrame(results)
print("\nBatch processing complete.")

Downloading...
From: https://drive.google.com/uc?id=1ZMnj0_rmXvLSFrNqYfMSPmxMEojhc_MH
To: /content/graphs_v2.pkl
100%|██████████| 6.63M/6.63M [00:00<00:00, 32.7MB/s]


Loaded 501 graphs
Processing graph 1/20...
Sample suggestion: Optimize op_108 by reducing its execution time, as it has a high negative slack and feeds into multiple critical paths. Consider using a more efficient algorithm or offloading to a specialized hardware unit.
Processing graph 2/20...
Processing graph 3/20...
Processing graph 4/20...
Processing graph 5/20...
Processing graph 6/20...
Processing graph 7/20...
Processing graph 8/20...
Processing graph 9/20...
Processing graph 10/20...
Processing graph 11/20...
Processing graph 12/20...
Processing graph 13/20...
Processing graph 14/20...
Processing graph 15/20...
Processing graph 16/20...
Processing graph 17/20...
Processing graph 18/20...
Processing graph 19/20...
Processing graph 20/20...

Batch processing complete.


In [20]:
summary_table = df_results[["graph_id", "predicted_bottleneck", "llm_bottlenecks", "avg_improvement_pct", "success", "recall_at_1"]]
print("Detailed Results per Graph:")
print(summary_table.to_string())

avg_improvement = df_results["avg_improvement_pct"].mean()
success_rate = df_results["success"].mean()
recall_at_1_rate = df_results["recall_at_1"].mean()

print("\n=== Aggregate Metrics ===")
print(f"Average ΔT improvement (%): {avg_improvement:.2f}%")
print(f"Success rate (≥2/3 suggestions improve): {success_rate:.2%}")
print(f"Recall@1 (top predicted bottleneck matches ground truth): {recall_at_1_rate:.2%}")

df_results.to_csv("/tmp/graphrag_results.csv", index=False)
print("Results saved to /tmp/graphrag_results.csv")

Detailed Results per Graph:
    graph_id predicted_bottleneck           llm_bottlenecks  avg_improvement_pct  success  recall_at_1
0          0               op_108  [op_108, op_126, op_167]             0.000009    False        False
1          1               op_167  [op_167, op_168, op_169]            -0.000002    False        False
2          2               op_100  [op_100, op_104, op_107]            -0.000040     True        False
3          3               op_174  [op_165, op_166, op_167]             0.000000    False        False
4          4               op_104  [op_104, op_100, op_107]             0.000779     True        False
5          5               op_182  [op_165, op_121, op_100]             0.000043     True        False
6          6               op_107  [op_100, op_107, op_104]             0.000246     True        False
7          7               op_132  [op_130, op_132, op_133]             0.000000    False        False
8          8               op_168  [op_167, o